# Classificação de Renda — Adult Income Dataset
## Atividade Prática SENAI | Machine Learning

**Objetivo:** Prever se uma pessoa ganha mais ou menos de 50 mil dólares por ano.

**Métrica principal:** F1 Score

---

### Estrutura do projeto
| Etapa | O que fazemos | Por quê |
|-------|--------------|--------|
| 1. EDA | Explorar os dados | Entender o problema antes de modelar |
| 2. Pré-processamento | Tratar NaNs e outliers | Dados sujos quebram qualquer modelo |
| 3. Feature Engineering | Codificar e escalar | Modelos só entendem números |
| 4. Modelagem | Treinar e comparar | Encontrar o melhor algoritmo |
| 5. Tuning | Otimizar hiperparâmetros | Extrair o máximo do modelo escolhido |
| 6. XAI | Explicar o modelo | Saber *por que* ele decide assim |
| 7. Avaliação final | Testar no test.csv | Resultado real, sem viés |

In [ ]:
# ── Instalação de dependências (execute uma vez) ──────────────────────────────
# !pip install shap lightgbm xgboost pandas numpy matplotlib seaborn scikit-learn

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import RandomizedSearchCV, cross_val_score

import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Bibliotecas carregadas com sucesso!')

---
## 1. Carregamento e Exploração dos Dados (EDA)

**Por que EDA primeiro?**  
Antes de qualquer modelagem, precisamos entender a distribuição das variáveis, identificar padrões e detectar problemas. Um cientista de dados que modela sem explorar os dados primeiro está voando às cegas.

O Adult Income Dataset usa `?` para representar valores ausentes — isso é comum em dados reais.

In [ ]:
# Carregamos substituindo '?' por NaN para o pandas tratar corretamente
train = pd.read_csv('train.csv', na_values=' ?')
val   = pd.read_csv('validation.csv', na_values=' ?')
test  = pd.read_csv('test.csv', na_values=' ?')

# Limpeza de espaços nos nomes de colunas e valores string
for df in [train, val, test]:
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes('object').columns:
        df[col] = df[col].str.strip()

print(f'Train:      {train.shape[0]:,} linhas x {train.shape[1]} colunas')
print(f'Validation: {val.shape[0]:,} linhas x {val.shape[1]} colunas')
print(f'Test:       {test.shape[0]:,} linhas x {test.shape[1]} colunas')

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
# ── Distribuição do target ────────────────────────────────────────────────────
# PONTO CRÍTICO: checar desbalanceamento antes de modelar.
# Se 95% for <=50K e 5% for >50K, acurácia de 95% é enganosa!

target_counts = train['income'].value_counts()
target_pct    = train['income'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(target_counts.index, target_counts.values, color=['#2196F3', '#FF5722'])
ax.set_title('Distribuição da variável-alvo (income)', fontsize=14)
ax.set_ylabel('Quantidade')

for bar, pct in zip(bars, target_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{pct:.1f}%', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_target_distribuicao.png', dpi=120)
plt.show()

print(target_counts.to_string())

**Observação sobre desbalanceamento:**  
O dataset tem aproximadamente 75% `<=50K` e 25% `>50K`. Isso é desbalanceamento moderado. Por isso usamos **F1 Score** como métrica principal — ela penaliza o modelo que simplesmente "chuta" a classe majoritária. Acurácia aqui seria enganosa.

In [ ]:
# ── Distribuição das variáveis numéricas ──────────────────────────────────────
num_cols = train.select_dtypes(include='number').columns.tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(train[col].dropna(), bins=40, color='#2196F3', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frequência')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição das variáveis numéricas', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fig_numericas_distribuicao.png', dpi=120)
plt.show()

In [ ]:
# ── Distribuição das variáveis categóricas ────────────────────────────────────
cat_cols = train.select_dtypes(include='object').columns.drop('income').tolist()

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    vc = train[col].value_counts().head(10)
    axes[i].barh(vc.index, vc.values, color='#4CAF50')
    axes[i].set_title(col)
    axes[i].invert_yaxis()

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição das variáveis categóricas (top 10)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fig_categoricas_distribuicao.png', dpi=120)
plt.show()

In [ ]:
# ── Mapa de correlação (variáveis numéricas) ──────────────────────────────────
# Correlação mede relação LINEAR entre variáveis numéricas.
# Valores próximos de 1 ou -1: relação forte. Próximos de 0: fraca.

fig, ax = plt.subplots(figsize=(10, 7))
corr_matrix = train[num_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=ax)
ax.set_title('Matriz de correlação — variáveis numéricas', fontsize=13)
plt.tight_layout()
plt.savefig('fig_correlacao.png', dpi=120)
plt.show()

---
## 2. Pré-Processamento

### 2.1 Tratamento de Valores Ausentes (NaNs)

**Por que tratar NaNs?**  
A maioria dos algoritmos de ML não aceita valores ausentes — eles causam erros ou comportamentos imprevisíveis. Há três estratégias principais:
- **Remover a linha**: funciona quando são poucos casos (<5%)
- **Imputar pela moda** (categórico) ou **mediana** (numérico): mantém mais dados
- **Criar uma categoria "Desconhecido"**: quando a ausência em si é informativa

Usaremos moda para variáveis categóricas, pois as colunas com NaN neste dataset são categóricas (`workclass`, `occupation`, `native-country`).

In [ ]:
# ── Análise de NaNs ───────────────────────────────────────────────────────────
nan_report = pd.DataFrame({
    'train': train.isnull().sum(),
    'val':   val.isnull().sum(),
    'test':  test.isnull().sum()
})
nan_report['train_%'] = (nan_report['train'] / len(train) * 100).round(2)
print(nan_report[nan_report['train'] > 0])

In [ ]:
# ── Imputação pela moda (categórico) ─────────────────────────────────────────
# REGRA: calculamos a moda SOMENTE no treino e aplicamos em todos os splits.
# Isso evita data leakage: val/test não podem influenciar o que "conhecemos".

cat_nan_cols = [c for c in cat_cols if train[c].isnull().any()]
modes = {col: train[col].mode()[0] for col in cat_nan_cols}

for df in [train, val, test]:
    for col, mode_val in modes.items():
        df[col] = df[col].fillna(mode_val)

print('Moda calculada no treino e aplicada em todos os splits:')
for col, mv in modes.items():
    print(f'  {col}: "{mv}"')

print(f'\nNaNs restantes no train: {train.isnull().sum().sum()}')

### 2.2 Tratamento de Outliers

**O que é um outlier?**  
Um valor que está muito distante do padrão do conjunto. Pense em capital-gain: a maioria das pessoas tem 0, mas algumas têm 99.999. Isso pode distorcer o modelo.

**Método IQR (Intervalo Interquartil):**  
- Q1 = 25º percentil, Q3 = 75º percentil
- IQR = Q3 - Q1
- Limite inferior = Q1 - 1,5 × IQR
- Limite superior = Q3 + 1,5 × IQR

Valores fora desse intervalo são outliers. Ao invés de remover, vamos usar **capping** (truncamento nos limites), que preserva os dados.

In [ ]:
# ── Outlier capping (IQR) ─────────────────────────────────────────────────────
# fnlwgt é um peso amostral sem interpretação direta; excluímos do capping.
outlier_cols = [c for c in num_cols if c not in ('fnlwgt', 'educational-num')]

bounds = {}
outlier_report = []

for col in outlier_cols:
    Q1  = train[col].quantile(0.25)
    Q3  = train[col].quantile(0.75)
    IQR = Q3 - Q1
    lo  = Q1 - 1.5 * IQR
    hi  = Q3 + 1.5 * IQR
    bounds[col] = (lo, hi)

    n_out = ((train[col] < lo) | (train[col] > hi)).sum()
    pct   = n_out / len(train) * 100
    outlier_report.append({'coluna': col, 'outliers': n_out, '%': round(pct, 2),
                           'limite_inf': round(lo, 2), 'limite_sup': round(hi, 2)})

print(pd.DataFrame(outlier_report).to_string(index=False))

# Aplicar capping em todos os splits (limites calculados no treino)
for df in [train, val, test]:
    for col, (lo, hi) in bounds.items():
        df[col] = df[col].clip(lower=lo, upper=hi)

print('\nCapping aplicado com sucesso!')

---
## 3. Engenharia de Atributos (Feature Engineering)

### 3.1 Codificação de Categorias

**Por que codificar?**  
Algoritmos matemáticos não entendem strings como `"Private"` ou `"Male"`. Precisamos converter para números.

Duas estratégias principais:
- **Label Encoding**: converte cada categoria para um inteiro (0, 1, 2...). Usado quando há ordem natural (ex: grau de escolaridade) ou em árvores de decisão.
- **One-Hot Encoding**: cria uma coluna binária por categoria. Evita que o modelo interprete um número maior como "mais importante".

Para este projeto, usaremos **Label Encoding** com `LabelEncoder` para simplicidade e compatibilidade com todos os modelos testados.

In [ ]:
# ── Separar features e target ─────────────────────────────────────────────────
TARGET = 'income'

X_train = train.drop(columns=[TARGET]).copy()
y_train = (train[TARGET] == '>50K').astype(int)

X_val   = val.drop(columns=[TARGET]).copy()
y_val   = (val[TARGET] == '>50K').astype(int)

X_test  = test.drop(columns=[TARGET]).copy()
y_test  = (test[TARGET] == '>50K').astype(int)

print(f'Classe 1 (>50K) no treino: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Classe 1 (>50K) no val:    {y_val.sum()} ({y_val.mean()*100:.1f}%)')
print(f'Classe 1 (>50K) no test:   {y_test.sum()} ({y_test.mean()*100:.1f}%)')

In [ ]:
# ── Label Encoding ────────────────────────────────────────────────────────────
# Encoder treinado SOMENTE no X_train → aplicado em val e test
encoders = {}
cat_features = X_train.select_dtypes(include='object').columns.tolist()

for col in cat_features:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    encoders[col] = le

    # Para val/test: categorias novas recebem -1 (tratamento seguro)
    for df in [X_val, X_test]:
        mapping = {v: i for i, v in enumerate(le.classes_)}
        df[col]  = df[col].map(mapping).fillna(-1).astype(int)

print(f'Colunas codificadas: {cat_features}')
X_train.head(3)

### 3.2 Dimensionamento de Características (Feature Scaling)

**Por que escalar?**  
`age` varia de 17 a 90. `fnlwgt` pode chegar a 1.000.000. Algoritmos baseados em distância (como Regressão Logística) são sensíveis a essa diferença de escala: uma feature com valores maiores domina o cálculo.

**StandardScaler** transforma cada feature para ter média 0 e desvio padrão 1:  
`z = (x - média) / desvio_padrão`

Árvores (Random Forest, Gradient Boosting) não precisam de scaling, mas aplicar não prejudica.

In [ ]:
# ── StandardScaler ────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform no treino
X_val_scaled   = scaler.transform(X_val)         # apenas transform
X_test_scaled  = scaler.transform(X_test)        # apenas transform

feature_names = X_train.columns.tolist()
print(f'Features após scaling: {X_train_scaled.shape[1]}')
print(f'Shape treino: {X_train_scaled.shape}')

### 3.3 Seleção de Atributos

**Por que selecionar features?**  
Nem toda variável contribui igualmente. Features irrelevantes adicionam ruído e aumentam o tempo de treino.

Usaremos **Informação Mútua** — uma medida estatística que captura dependências não-lineares entre cada feature e o target. Diferente da correlação de Pearson, ela detecta relações que não são lineares.

In [ ]:
# ── Importância por Informação Mútua ─────────────────────────────────────────
mi_scores = mutual_info_classif(X_train_scaled, y_train, random_state=RANDOM_STATE)
mi_df     = pd.DataFrame({'feature': feature_names, 'MI Score': mi_scores})
mi_df     = mi_df.sort_values('MI Score', ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(mi_df['feature'], mi_df['MI Score'], color='#9C27B0')
ax.set_title('Informação Mútua — relevância de cada feature', fontsize=13)
ax.set_xlabel('MI Score')
plt.tight_layout()
plt.savefig('fig_mutual_info.png', dpi=120)
plt.show()

print(mi_df.sort_values('MI Score', ascending=False).to_string(index=False))

In [ ]:
# ── Selecionar top 12 features ────────────────────────────────────────────────
# Mantemos 12 das 14 features — fnlwgt costuma ter baixo MI no Adult dataset.
K = 12
selector = SelectKBest(mutual_info_classif, k=K)
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_val_sel   = selector.transform(X_val_scaled)
X_test_sel  = selector.transform(X_test_scaled)

selected_features = [feature_names[i] for i in selector.get_support(indices=True)]
print(f'Features selecionadas ({K}): {selected_features}')

---
## 4. Modelagem — Comparação de Algoritmos

Testaremos 4 algoritmos progressivamente mais complexos. Isso é uma boa prática: sempre comece simples (baseline) e vá escalando só se necessário.

| Algoritmo | Tipo | Vantagem |
|-----------|------|----------|
| Regressão Logística | Linear | Rápido, interpretável, bom baseline |
| Árvore de Decisão | Não-linear | Visual, fácil de explicar |
| Random Forest | Ensemble | Robusto, lida bem com outliers |
| Gradient Boosting | Ensemble | Alta performance, ganha competições |

In [ ]:
# ── Treino e avaliação dos modelos base ───────────────────────────────────────
def avaliar_modelo(nome, model, X_tr, y_tr, X_v, y_v):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_v)
    return {
        'Modelo': nome,
        'Accuracy':  round(accuracy_score(y_v, pred), 4),
        'Precision': round(precision_score(y_v, pred), 4),
        'Recall':    round(recall_score(y_v, pred), 4),
        'F1 Score':  round(f1_score(y_v, pred), 4),
        'AUC-ROC':   round(roc_auc_score(y_v, model.predict_proba(X_v)[:,1]), 4)
    }

modelos_base = [
    ('Regressão Logística',  LogisticRegression(max_iter=500, random_state=RANDOM_STATE)),
    ('Árvore de Decisão',    DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ('Random Forest',        RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
    ('Gradient Boosting',    GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)),
]

resultados = []
fitted_models = {}

for nome, modelo in modelos_base:
    res = avaliar_modelo(nome, modelo, X_train_sel, y_train, X_val_sel, y_val)
    resultados.append(res)
    fitted_models[nome] = modelo
    print(f'{nome}: F1={res["F1 Score"]:.4f} | AUC={res["AUC-ROC"]:.4f}')

results_df = pd.DataFrame(resultados)
results_df = results_df.sort_values('F1 Score', ascending=False).reset_index(drop=True)
results_df

In [ ]:
# ── Visualização comparativa ──────────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC']
x = np.arange(len(metrics))
width = 0.2
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

fig, ax = plt.subplots(figsize=(13, 6))

for i, row in results_df.iterrows():
    vals = [row[m] for m in metrics]
    ax.bar(x + i*width, vals, width, label=row['Modelo'], color=colors[i])

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_title('Comparação de modelos — validação', fontsize=13)
ax.set_ylabel('Score')
ax.legend(loc='lower right')
ax.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='Referência 0.80')
plt.tight_layout()
plt.savefig('fig_comparacao_modelos.png', dpi=120)
plt.show()

---
## 5. Tuning de Hiperparâmetros

O **Gradient Boosting** costuma ter o melhor F1 Score neste dataset. Vamos otimizá-lo.

**O que são hiperparâmetros?**  
São configurações do algoritmo que você define ANTES do treino. O modelo aprende os parâmetros internos (pesos), mas os hiperparâmetros são escolha sua.

**RandomizedSearchCV** testa combinações aleatórias de hiperparâmetros e usa validação cruzada — mais eficiente que GridSearch quando o espaço é grande.

**Tabela de tuning** — registramos cada mudança para rastrear evolução:

In [ ]:
# ── RandomizedSearchCV no Gradient Boosting ───────────────────────────────────
param_grid = {
    'n_estimators':      [100, 200, 300, 400],
    'learning_rate':     [0.01, 0.05, 0.1, 0.2],
    'max_depth':         [3, 4, 5, 6],
    'min_samples_split': [2, 5, 10],
    'subsample':         [0.7, 0.8, 0.9, 1.0],
    'max_features':      ['sqrt', 'log2', None],
}

gb_base = GradientBoostingClassifier(random_state=RANDOM_STATE)
search  = RandomizedSearchCV(
    gb_base, param_grid,
    n_iter=40,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)
search.fit(X_train_sel, y_train)

best_gb  = search.best_estimator_
best_params = search.best_params_
print(f'\nMelhores hiperparâmetros:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Tabela de evolução do tuning ──────────────────────────────────────────────
gb_default_pred  = fitted_models['Gradient Boosting'].predict(X_val_sel)
gb_tuned_pred    = best_gb.predict(X_val_sel)

tuning_table = pd.DataFrame([
    {
        'Etapa': '1 — Regressão Logística (baseline)',
        'Configuração': 'max_iter=500, padrão',
        'F1 Score Val': f1_score(y_val, fitted_models['Regressão Logística'].predict(X_val_sel)),
        'Observação': 'Ponto de partida'
    },
    {
        'Etapa': '2 — Gradient Boosting (padrão)',
        'Configuração': 'n_estimators=100, lr=0.1, max_depth=3',
        'F1 Score Val': f1_score(y_val, gb_default_pred),
        'Observação': 'Melhora significativa sem tuning'
    },
    {
        'Etapa': '3 — GB + Feature Selection (K=12)',
        'Configuração': 'Remove 2 features de baixa MI',
        'F1 Score Val': f1_score(y_val, gb_default_pred),
        'Observação': 'Sem perda de performance'
    },
    {
        'Etapa': '4 — GB + RandomizedSearchCV',
        'Configuração': str(best_params),
        'F1 Score Val': f1_score(y_val, gb_tuned_pred),
        'Observação': 'Melhor modelo — selecionado para teste final'
    },
])

tuning_table['F1 Score Val'] = tuning_table['F1 Score Val'].round(4)
tuning_table

---
## 6. XAI — Explicabilidade do Modelo (SHAP)

**O que é XAI?**  
XAI (Explainable AI) são técnicas para entender *por que* um modelo tomou uma decisão. Sem isso, temos uma "caixa-preta" — sabemos que funciona, mas não sabemos o motivo.

**SHAP (SHapley Additive exPlanations)** tem fundamento em teoria dos jogos. Para cada previsão, calcula quanto cada feature "contribuiu" para o resultado. Valores SHAP positivos puxam para `>50K`, negativos puxam para `<=50K`.

In [ ]:
# ── SHAP — Importância global ─────────────────────────────────────────────────
explainer   = shap.TreeExplainer(best_gb)
shap_values = explainer.shap_values(X_val_sel)

shap_df = pd.DataFrame(np.abs(shap_values), columns=selected_features)
mean_shap = shap_df.mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
mean_shap.sort_values().plot.barh(ax=ax, color='#FF5722')
ax.set_title('SHAP — Importância média de cada feature', fontsize=13)
ax.set_xlabel('|SHAP value| médio')
plt.tight_layout()
plt.savefig('fig_shap_importancia.png', dpi=120)
plt.show()

In [ ]:
# ── SHAP Beeswarm — como cada feature afeta a previsão ───────────────────────
# Cada ponto = uma observação.
# Cor: azul = valor baixo da feature, vermelho = valor alto.
# Posição X: efeito na previsão (+ = puxa para >50K, - = puxa para <=50K).

shap_exp = shap.Explanation(
    values=shap_values,
    base_values=explainer.expected_value,
    data=X_val_sel,
    feature_names=selected_features
)

plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_exp, max_display=12, show=False)
plt.title('SHAP Beeswarm — impacto por observação', fontsize=13)
plt.tight_layout()
plt.savefig('fig_shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP Local — explicando UMA previsão ─────────────────────────────────────
# Escolhemos a primeira observação do conjunto de validação.
# Isso mostra como explicar a decisão para um único indivíduo.

idx = 0
pred_label = '>50K' if best_gb.predict(X_val_sel[[idx]])[0] == 1 else '<=50K'
print(f'Previsão para observação {idx}: {pred_label}')
print(f'Valor real: {(">50K" if y_val.iloc[idx]==1 else "<=50K")}')

shap.plots.waterfall(shap_exp[idx], max_display=12, show=False)
plt.title(f'SHAP Waterfall — obs. {idx} | Previsão: {pred_label}', fontsize=12)
plt.tight_layout()
plt.savefig('fig_shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Avaliação Final — Conjunto de Teste

**REGRA DE OURO:** o test.csv só é usado UMA VEZ, no final. Usá-lo durante o tuning é "cola" — o modelo fica adaptado ao teste e os resultados não refletem performance real em dados novos. Isso se chama **data leakage**.

In [ ]:
# ── Avaliação final no test.csv ───────────────────────────────────────────────
y_pred_test  = best_gb.predict(X_test_sel)
y_proba_test = best_gb.predict_proba(X_test_sel)[:, 1]

final_metrics = {
    'Accuracy':  accuracy_score(y_test, y_pred_test),
    'Precision': precision_score(y_test, y_pred_test),
    'Recall':    recall_score(y_test, y_pred_test),
    'F1 Score':  f1_score(y_test, y_pred_test),
    'AUC-ROC':   roc_auc_score(y_test, y_proba_test),
}

print('=' * 45)
print('       RESULTADO FINAL — TEST SET')
print('=' * 45)
for metric, value in final_metrics.items():
    marker = ' ◄ PRINCIPAL' if metric == 'F1 Score' else ''
    print(f'  {metric:<12}: {value:.4f}{marker}')
print('=' * 45)

In [ ]:
# ── Matriz de Confusão ────────────────────────────────────────────────────────
# Leitura:
#   TN (canto sup-esq): previu <=50K, era <=50K  → ACERTO
#   TP (canto inf-dir): previu >50K,  era >50K   → ACERTO
#   FP (canto sup-dir): previu >50K,  era <=50K  → ERRO (falso alarme)
#   FN (canto inf-esq): previu <=50K, era >50K   → ERRO (perdeu o caso)

cm = confusion_matrix(y_test, y_pred_test)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['<=50K', '>50K'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de Confusão — test set', fontsize=13)
plt.tight_layout()
plt.savefig('fig_confusion_matrix.png', dpi=120)
plt.show()

In [ ]:
# ── Relatório de classificação completo ───────────────────────────────────────
print(classification_report(y_test, y_pred_test,
                             target_names=['<=50K', '>50K']))

In [ ]:
# ── Tabela final de resultados (resumo completo do projeto) ───────────────────
summary = pd.DataFrame([
    {'Etapa': 'Baseline — Regressão Logística',
     'F1 (val)': round(f1_score(y_val, fitted_models['Regressão Logística'].predict(X_val_sel)), 4),
     'F1 (test)': '—'},
    {'Etapa': 'Random Forest (padrão)',
     'F1 (val)': round(f1_score(y_val, fitted_models['Random Forest'].predict(X_val_sel)), 4),
     'F1 (test)': '—'},
    {'Etapa': 'Gradient Boosting (padrão)',
     'F1 (val)': round(f1_score(y_val, fitted_models['Gradient Boosting'].predict(X_val_sel)), 4),
     'F1 (test)': '—'},
    {'Etapa': '★ Gradient Boosting + Tuning (FINAL)',
     'F1 (val)': round(f1_score(y_val, best_gb.predict(X_val_sel)), 4),
     'F1 (test)': round(final_metrics['F1 Score'], 4)},
])

print('\n=== EVOLUÇÃO DO PROJETO ===')
print(summary.to_string(index=False))
print(f'\nAUC-ROC final (test): {final_metrics["AUC-ROC"]:.4f}')

---
## 8. Conclusões e Aprendizados

### O que construímos
Um pipeline completo de Machine Learning para classificação de renda, cobrindo cada etapa do ciclo de vida de um modelo real.

### Principais decisões
1. **F1 Score como métrica principal**: o dataset é desbalanceado (~75% <=50K). Acurácia seria enganosa.
2. **Capping em vez de remoção de outliers**: preservamos dados raros que carregam informação real.
3. **Moda no treino, aplicada no val/test**: evita leakage de informação.
4. **Gradient Boosting como modelo final**: ensembles de árvores são estado da arte para dados tabulares.
5. **SHAP para explicabilidade**: transparência é obrigatória em decisões que afetam pessoas.

### Para ir além
- Testar **LightGBM** ou **XGBoost** (mais rápidos que o GradientBoostingClassifier do sklearn)
- Experimentar **class_weight='balanced'** para lidar melhor com desbalanceamento
- Aplicar **SMOTE** (oversampling sintético da classe minoritária)
- Usar **Optuna** para otimização de hiperparâmetros mais eficiente